# Music Annotation Project 60R3.1
## Read-only Page258 component-renumbering invariance audit

**Runtime:** CPU. Select **Runtime → Run all**.

This notebook tests whether the frozen 59R1 REMOVE estimator changes its Page258 scores and decisions when the arbitrary numeric labels in `component_id` are changed while component membership, geometry, all other predictors, the model, the calibrator, the 0.995 threshold, and the page-global coherence rule remain fixed.

### Primary hypothesis

A connected-component ID is an identifier, not component geometry. A frozen model that depends on its numeric value is not invariant to semantically equivalent component renumbering.

### Read-only contract

- no model fitting or retraining;
- no calibrator fitting or recalibration;
- no threshold change;
- no change to the original 60R2 Page258 output;
- no change to component pixel membership;
- no change to `mask_depth`, `boundary_distance`, or any other non-target predictor unless that predictor is the named counterfactual under test;
- source artifacts are SHA-256 checked before and after the audit;
- outputs are written only to a new timestamped 60R3.1 directory.

### Required headline table

The notebook writes `tables/page258_geometry_disaggregation_table.csv` with:

| Audit | Mean raw-score movement on baseline M5 | Pixels crossing 0.995 upward | Coherent SAFE_REMOVE delta |
|---|---:|---:|---:|
| `component_id` median substitution | measured | measured | measured |
| `component_id` component-level permutation | measured over fixed seeds | measured | measured |
| `component_id` area-order renumbering | measured | measured | measured |
| `mask_depth` only | measured | measured | measured |
| `boundary_distance` only | measured | measured | measured |
| depth + distance | measured | measured | measured |
| original three-feature group | measured | measured | measured; expected replay target 164,184 if the same 60R3 source/reference procedure is resolved |

The known `164,184` value is treated as a **replay target**, not hard-coded as a result.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q pandas numpy pyarrow scikit-learn joblib opencv-python-headless matplotlib

## 1. Configuration, resolution rules, and helpers

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, shutil, random, sys, platform, zipfile, math

import cv2
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
PAGE = 'Page258'

# Optional exact run-root overrides.
SOURCE_60R3_RUN_OVERRIDE = None
SOURCE_60R2_RUN_OVERRIDE = None
REMOVE_59R1_RUN_OVERRIDE = None

SOURCE_60R3_ROOTS = [ROOT/'page258_shap_counterfactual_audit']
SOURCE_60R2_ROOTS = [
    ROOT/'notebook60r2_verified_integrated_runtime',
    ROOT/'notebook60r2_integrated_runtime',
]
REMOVE_59R1_ROOTS = [
    ROOT/'observable_residual_mask_m5_v8_7_deployment_freeze',
    ROOT/'remove_deployment_freeze_v8_8',
    ROOT/'remove_deployment_fit_and_freeze_v8_8',
]

OUTPUT_ROOT = ROOT/'page258_component_renumbering_invariance_audit'
RUN = OUTPUT_ROOT/(datetime.now().strftime('%Y%m%d_%H%M%S') + '_60R3_1')
TABLES = RUN/'tables'; AUDIT = RUN/'audit'; VISUALS = RUN/'visuals'; UPLOAD = RUN/'upload'
for p in [RUN,TABLES,AUDIT,VISUALS,UPLOAD]: p.mkdir(parents=True,exist_ok=False)

SEED = 42
PERMUTATION_SEEDS = [11, 23, 37, 53, 71, 89, 107, 131, 151, 173]
EXPECTED_ORIGINAL_THREE_FEATURE_SAFE_DELTA = 164184
COUNTERFACTUAL_OBSERVED_BINS = [0,64,96,128,160,192,224,256]


def require(condition, message):
    if not bool(condition): raise RuntimeError(message)


def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1048576),b''): h.update(block)
    return h.hexdigest()


def read_json(path):
    path=Path(path); require(path.is_file(),f'Missing JSON: {path}')
    value=json.loads(path.read_text()); require(isinstance(value,dict),f'Invalid JSON root: {path}')
    return value


def atomic_json(path,value):
    Path(path).write_text(json.dumps(value,indent=2,sort_keys=True,default=str))


def latest_complete(roots, override, required):
    if override is not None:
        run=Path(override); run=run.parent if run.name=='bundle' else run
        require(all((run/r).is_file() for r in required),f'Incomplete override: {run}')
        return run
    hits=[]
    for root in roots:
        if root.exists():
            for candidate in root.iterdir():
                if candidate.is_dir() and all((candidate/r).is_file() for r in required): hits.append(candidate)
    require(hits,f'No complete run below {[str(r) for r in roots]}')
    return max(hits,key=lambda p:p.stat().st_mtime)

print('Output:',RUN)

## 2. Resolve the prior 60R3 data, frozen 60R2 Page258 output, and frozen 59R1 bundle

In [ ]:
R3_REQUIRED = [
    'tables/page258_frozen_rows.parquet',
    'tables/page258_counterfactual_ablation.csv',
    'page258_attribution_report.json',
    'source_hashes_before.json',
]
R2_REQUIRED = [
    'runtime_manifest.json','verdict.json','tables/pixel_predictions.parquet',
    f'pages/{PAGE}/observed.png',f'pages/{PAGE}/safe_remove.png',f'pages/{PAGE}/m5_remove_mask.png',
]
RM_REQUIRED = [
    'bundle/remove_model.joblib','bundle/remove_calibrator.joblib','bundle/predictor_schema.json',
    'bundle/policy.json','bundle/freeze_manifest.json','replay/replay_test.json',
]

SOURCE_60R3=latest_complete(SOURCE_60R3_ROOTS,SOURCE_60R3_RUN_OVERRIDE,R3_REQUIRED)
SOURCE_60R2=latest_complete(SOURCE_60R2_ROOTS,SOURCE_60R2_RUN_OVERRIDE,R2_REQUIRED)
REMOVE_RUN=latest_complete(REMOVE_59R1_ROOTS,REMOVE_59R1_RUN_OVERRIDE,RM_REQUIRED)
REMOVE_BUNDLE=REMOVE_RUN/'bundle'

paths={
 'r3_rows':SOURCE_60R3/'tables'/'page258_frozen_rows.parquet',
 'r3_group_cf':SOURCE_60R3/'tables'/'page258_counterfactual_ablation.csv',
 'r3_report':SOURCE_60R3/'page258_attribution_report.json',
 'r2_pixels':SOURCE_60R2/'tables'/'pixel_predictions.parquet',
 'observed':SOURCE_60R2/'pages'/PAGE/'observed.png',
 'safe_remove':SOURCE_60R2/'pages'/PAGE/'safe_remove.png',
 'm5_remove':SOURCE_60R2/'pages'/PAGE/'m5_remove_mask.png',
 'model':REMOVE_BUNDLE/'remove_model.joblib',
 'calibrator':REMOVE_BUNDLE/'remove_calibrator.joblib',
 'schema':REMOVE_BUNDLE/'predictor_schema.json',
 'policy':REMOVE_BUNDLE/'policy.json',
 'freeze':REMOVE_BUNDLE/'freeze_manifest.json',
 'replay':REMOVE_RUN/'replay'/'replay_test.json',
}
source_hashes={k:sha256_file(v) for k,v in paths.items()}
atomic_json(AUDIT/'source_hashes_before.json',{k:{'path':str(paths[k]),'sha256':v} for k,v in source_hashes.items()})

schema=read_json(paths['schema']); policy=read_json(paths['policy']); replay=read_json(paths['replay'])
require(replay.get('passed') is True,'59R1 replay did not pass.')
predictors=schema.get('ordered_predictors') or schema.get('predictors')
require(isinstance(predictors,list) and len(predictors)==17,'Expected 17 frozen predictors.')
for required in ['component_id','mask_depth','boundary_distance']:
    require(required in predictors,f'Frozen schema lacks {required}.')

REMOVE_THRESHOLD=float(policy.get('remove_threshold',policy.get('threshold',.995)))
REMOVE_CONNECTIVITY=int(policy.get('connectivity',8))
REMOVE_MINIMUM_RUN=int(policy.get('minimum_run',policy.get('minimum_region_size',2)))
REMOVE_REGION_MEAN_THRESHOLD=float(policy.get('region_mean_threshold',REMOVE_THRESHOLD))

model=joblib.load(paths['model']); calibrator=joblib.load(paths['calibrator'])
require(hasattr(model,'predict_proba') and hasattr(calibrator,'predict'),'Frozen scoring objects invalid.')

print('60R3:',SOURCE_60R3)
print('60R2:',SOURCE_60R2)
print('59R1:',REMOVE_RUN)
print('Threshold:',REMOVE_THRESHOLD)

## 3. Replay Page258 exactly and reconstruct immutable component membership

In [ ]:
page=pd.read_parquet(paths['r3_rows']).copy().reset_index(drop=True)
require(len(page)>0 and set(['page','y','x','raw_remove_probability','p_remove','safe_remove']).issubset(page.columns),'Invalid Page258 frozen rows.')
require(set(page.page.astype(str))=={PAGE},'Frozen rows contain another page.')
missing=[c for c in predictors if c not in page.columns]; require(not missing,f'Missing predictors: {missing}')
X=page[predictors].apply(pd.to_numeric,errors='coerce')
raw0=np.asarray(model.predict_proba(X)[:,1],np.float64); p0=np.asarray(calibrator.predict(raw0),np.float64)
require(np.array_equal(raw0,page.raw_remove_probability.to_numpy(np.float64)),'Raw-score replay not exact.')
require(np.array_equal(p0,page.p_remove.to_numpy(np.float64)),'p_remove replay not exact.')

observed=cv2.imread(str(paths['observed']),0)
safe0=cv2.imread(str(paths['safe_remove']),0)>0
m50=cv2.imread(str(paths['m5_remove']),0)>0
require(observed is not None and observed.shape==safe0.shape==m50.shape,'Image shape mismatch.')
inside=np.zeros(observed.shape,bool); inside[page.y.to_numpy(int),page.x.to_numpy(int)]=True
require(np.array_equal(m50,inside&~safe0),'Baseline M5 identity mismatch.')

# Component membership from the frozen component_id column.
component_values=np.sort(page.component_id.astype(int).unique())
require((component_values>0).all(),'component_id must be positive for Page258 rows.')
component_membership={cid:np.flatnonzero(page.component_id.astype(int).to_numpy()==cid) for cid in component_values}
component_rows=[]
for cid,idx in component_membership.items():
    ys=page.y.to_numpy(int)[idx]; xs=page.x.to_numpy(int)[idx]
    component_rows.append({'original_component_id':int(cid),'pixels':len(idx),'x_min':int(xs.min()),'x_max':int(xs.max()),'y_min':int(ys.min()),'y_max':int(ys.max()),'centroid_x':float(xs.mean()),'centroid_y':float(ys.mean())})
components=pd.DataFrame(component_rows)
components.to_csv(TABLES/'page258_component_membership.csv',index=False)

# Immutable semantic membership hash excludes numeric labels.
def membership_hash():
    h=hashlib.sha256()
    for row in components.sort_values(['y_min','x_min','pixels']).itertuples(index=False):
        idx=component_membership[int(row.original_component_id)]
        coords=np.stack([page.y.to_numpy(np.int32)[idx],page.x.to_numpy(np.int32)[idx]],axis=1)
        coords=coords[np.lexsort((coords[:,1],coords[:,0]))]
        h.update(coords.tobytes())
    return h.hexdigest()
BASE_MEMBERSHIP_HASH=membership_hash()
print('Components:',len(components),'Membership hash:',BASE_MEMBERSHIP_HASH)

## 4. Frozen page-global coherence and scoring helpers

In [ ]:
def coherent_safe_remove(frame,p_remove,shape):
    provisional=np.asarray(p_remove)>=REMOVE_THRESHOLD
    candidate=np.zeros(shape,np.uint8)
    candidate[frame.loc[provisional,'y'].to_numpy(int),frame.loc[provisional,'x'].to_numpy(int)]=1
    n,labels,stats,_=cv2.connectedComponentsWithStats(candidate,REMOVE_CONNECTIVITY)
    pmap=np.full(shape,np.nan,np.float32)
    pmap[frame.y.to_numpy(int),frame.x.to_numpy(int)]=np.asarray(p_remove,np.float32)
    safe=np.zeros(shape,bool)
    for cid in range(1,n):
        region=labels==cid; values=pmap[region]; finite=values[np.isfinite(values)]
        passed=bool(int(region.sum())>=REMOVE_MINIMUM_RUN and len(finite)==int(region.sum()) and float(finite.mean())>=REMOVE_REGION_MEAN_THRESHOLD and float(finite.min())>=REMOVE_THRESHOLD)
        if passed:safe[region]=True
    return safe

require(np.array_equal(coherent_safe_remove(page,p0,observed.shape),safe0),'Frozen coherence replay not exact.')
baseline_m5_rows=~page.safe_remove.astype(bool).to_numpy()


def score_audit(name,Xcf,details=None):
    # Verify only allowed changed columns before scoring.
    raw=np.asarray(model.predict_proba(Xcf[predictors])[:,1],np.float64)
    prob=np.asarray(calibrator.predict(raw),np.float64)
    safe=coherent_safe_remove(page,prob,observed.shape)
    up=(p0<REMOVE_THRESHOLD)&(prob>=REMOVE_THRESHOLD)
    down=(p0>=REMOVE_THRESHOLD)&(prob<REMOVE_THRESHOLD)
    row={
      'audit':name,
      'mean_raw_score_movement_all':float(np.mean(raw-raw0)),
      'mean_raw_score_movement_baseline_M5':float(np.mean((raw-raw0)[baseline_m5_rows])),
      'median_raw_score_movement_baseline_M5':float(np.median((raw-raw0)[baseline_m5_rows])),
      'mean_p_remove_movement_baseline_M5':float(np.mean((prob-p0)[baseline_m5_rows])),
      'pixels_crossing_0_995_upward':int(up.sum()),
      'baseline_M5_pixels_crossing_0_995_upward':int((up&baseline_m5_rows).sum()),
      'pixels_crossing_0_995_downward':int(down.sum()),
      'coherent_SAFE_REMOVE_baseline':int(safe0.sum()),
      'coherent_SAFE_REMOVE_counterfactual':int(safe.sum()),
      'coherent_SAFE_REMOVE_delta':int(safe.sum()-safe0.sum()),
      'M5_counterfactual':int((inside&~safe).sum()),
      'M5_delta':int((inside&~safe).sum()-m50.sum()),
    }
    if details:row.update(details)
    return row,raw,prob,safe

## 5. Matched references for depth and distance counterfactuals

In [ ]:
reference=page.copy()
reference['observed_band']=pd.cut(reference.observed_value,COUNTERFACTUAL_OBSERVED_BINS,right=False,include_lowest=True,duplicates='drop')
safe_reference=reference[reference.safe_remove.astype(bool)]
require(len(safe_reference)>0,'No SAFE_REMOVE reference rows.')
band_median=safe_reference.groupby('observed_band',observed=True)[predictors].median(numeric_only=True)
overall_median=safe_reference[predictors].median(numeric_only=True)


def matched_replace(Xbase,features):
    out=Xbase.copy(); bands=reference.observed_band.reset_index(drop=True)
    for feature in features:
        mapped=bands.map(band_median[feature] if feature in band_median else {})
        out[feature]=mapped.astype(float).fillna(float(overall_median[feature]))
    return out

## 6. Single-feature and original grouped counterfactuals

In [ ]:
results=[]; saved={}
for name,features in [
    ('component_id median substitution',['component_id']),
    ('mask_depth only',['mask_depth']),
    ('boundary_distance only',['boundary_distance']),
    ('depth + distance',['mask_depth','boundary_distance']),
    ('original three-feature group',['component_id','mask_depth','boundary_distance']),
]:
    Xcf=matched_replace(X,features)
    row,raw,prob,safe=score_audit(name,Xcf,{'changed_features':'|'.join(features),'method':'matched SAFE_REMOVE median by observed-intensity band'})
    results.append(row);saved[name]=(raw,prob,safe)

# Check that the original group reproduces the earlier 60R3 result when source/reference semantics are the same.
original_delta=next(r['coherent_SAFE_REMOVE_delta'] for r in results if r['audit']=='original three-feature group')
original_group_replay_exact=(original_delta==EXPECTED_ORIGINAL_THREE_FEATURE_SAFE_DELTA)
print('Original group delta:',original_delta,'Expected:',EXPECTED_ORIGINAL_THREE_FEATURE_SAFE_DELTA,'Exact:',original_group_replay_exact)

## 7. Semantically equivalent component-level renumberings

In [ ]:
original_ids=components.original_component_id.astype(int).tolist()


def renumber_vector(mapping):
    values=page.component_id.astype(int).map(mapping)
    require(values.notna().all(),'Incomplete component mapping.')
    return values.astype(float).to_numpy()


def audit_mapping(name,mapping,method,seed=None):
    require(set(mapping)==set(original_ids),'Mapping source IDs incomplete.')
    require(len(set(mapping.values()))==len(original_ids),'Mapping is not bijective.')
    Xcf=X.copy(); Xcf['component_id']=renumber_vector(mapping)
    unchanged=[c for c in predictors if c!='component_id' and not np.array_equal(Xcf[c].to_numpy(),X[c].to_numpy())]
    require(not unchanged,f'Non-component predictors changed: {unchanged}')
    require(membership_hash()==BASE_MEMBERSHIP_HASH,'Component pixel membership changed.')
    return score_audit(name,Xcf,{'changed_features':'component_id','method':method,'seed':seed})

# Reverse labels.
reverse_mapping=dict(zip(original_ids,list(reversed(original_ids))))
row,raw,prob,safe=audit_mapping('component_id reverse renumbering',reverse_mapping,'bijective reverse label order')
results.append(row);saved[row['audit']]=(raw,prob,safe)

# Cyclic shift labels.
shifted=original_ids[1:]+original_ids[:1]
cyclic_mapping=dict(zip(original_ids,shifted))
row,raw,prob,safe=audit_mapping('component_id cyclic renumbering',cyclic_mapping,'bijective cyclic shift')
results.append(row);saved[row['audit']]=(raw,prob,safe)

# Area-order renumbering: largest component gets 1, next largest gets 2, etc.
area_order=components.sort_values(['pixels','y_min','x_min'],ascending=[False,True,True]).original_component_id.astype(int).tolist()
area_mapping={old:new for new,old in enumerate(area_order,1)}
row,raw,prob,safe=audit_mapping('component_id area-order renumbering',area_mapping,'bijective rank by descending component area')
results.append(row);saved[row['audit']]=(raw,prob,safe)

# Geometry-neutral centroid order variants are diagnostic additions.
yx_order=components.sort_values(['centroid_y','centroid_x']).original_component_id.astype(int).tolist()
xy_order=components.sort_values(['centroid_x','centroid_y']).original_component_id.astype(int).tolist()
for name,order,method in [
    ('component_id centroid-yx renumbering',yx_order,'bijective rank by centroid y then x'),
    ('component_id centroid-xy renumbering',xy_order,'bijective rank by centroid x then y'),
]:
    mapping={old:new for new,old in enumerate(order,1)}
    row,raw,prob,safe=audit_mapping(name,mapping,method)
    results.append(row);saved[name]=(raw,prob,safe)

# Multiple fixed-seed component-level permutations.
permutation_rows=[]
for seed in PERMUTATION_SEEDS:
    rng=np.random.default_rng(seed); perm=original_ids.copy(); rng.shuffle(perm)
    mapping=dict(zip(original_ids,perm))
    name=f'component_id component-level permutation seed {seed}'
    row,raw,prob,safe=audit_mapping(name,mapping,'bijective random component-level permutation',seed)
    permutation_rows.append(row);saved[name]=(raw,prob,safe)
results.extend(permutation_rows)
pd.DataFrame(permutation_rows).to_csv(TABLES/'component_id_permutation_replicates.csv',index=False)

## 8. Requested headline table

In [ ]:
all_results=pd.DataFrame(results)
all_results.to_csv(TABLES/'all_counterfactual_results.csv',index=False)

perm=pd.DataFrame(permutation_rows)
perm_summary={
 'audit':'component_id component-level permutation',
 'raw_score_movement':float(perm.mean_raw_score_movement_baseline_M5.mean()),
 'raw_score_movement_min':float(perm.mean_raw_score_movement_baseline_M5.min()),
 'raw_score_movement_max':float(perm.mean_raw_score_movement_baseline_M5.max()),
 'pixels_crossing_0_995':float(perm.baseline_M5_pixels_crossing_0_995_upward.mean()),
 'pixels_crossing_0_995_min':int(perm.baseline_M5_pixels_crossing_0_995_upward.min()),
 'pixels_crossing_0_995_max':int(perm.baseline_M5_pixels_crossing_0_995_upward.max()),
 'coherent_SAFE_REMOVE_delta':float(perm.coherent_SAFE_REMOVE_delta.mean()),
 'coherent_SAFE_REMOVE_delta_min':int(perm.coherent_SAFE_REMOVE_delta.min()),
 'coherent_SAFE_REMOVE_delta_max':int(perm.coherent_SAFE_REMOVE_delta.max()),
 'replicates':len(perm),
}

lookup=all_results.set_index('audit')
requested=[]
for audit in [
 'component_id median substitution',
 'component_id area-order renumbering',
 'mask_depth only',
 'boundary_distance only',
 'depth + distance',
 'original three-feature group',
]:
    r=lookup.loc[audit]
    requested.append({
      'Audit':audit,
      'Raw-score movement':float(r.mean_raw_score_movement_baseline_M5),
      'Pixels crossing 0.995':int(r.baseline_M5_pixels_crossing_0_995_upward),
      'Coherent SAFE_REMOVE delta':int(r.coherent_SAFE_REMOVE_delta),
      'Range / replay note':('expected prior grouped delta 164184; exact='+str(original_group_replay_exact)) if audit=='original three-feature group' else '',
    })
requested.insert(1,{
 'Audit':perm_summary['audit'],
 'Raw-score movement':perm_summary['raw_score_movement'],
 'Pixels crossing 0.995':perm_summary['pixels_crossing_0_995'],
 'Coherent SAFE_REMOVE delta':perm_summary['coherent_SAFE_REMOVE_delta'],
 'Range / replay note':f"{perm_summary['replicates']} seeds; delta range {perm_summary['coherent_SAFE_REMOVE_delta_min']} to {perm_summary['coherent_SAFE_REMOVE_delta_max']}",
})
headline=pd.DataFrame(requested)
headline.to_csv(TABLES/'page258_geometry_disaggregation_table.csv',index=False)
display(headline)

## 9. Invariance verdict

In [ ]:
renumber_names=[
 'component_id reverse renumbering','component_id cyclic renumbering',
 'component_id area-order renumbering','component_id centroid-yx renumbering','component_id centroid-xy renumbering',
]+[f'component_id component-level permutation seed {s}' for s in PERMUTATION_SEEDS]
renumber=all_results[all_results.audit.isin(renumber_names)].copy()
any_raw_change=bool((renumber.mean_raw_score_movement_all.abs()>0).any())
any_threshold_change=bool(((renumber.pixels_crossing_0_995_upward+renumber.pixels_crossing_0_995_downward)>0).any())
any_coherent_change=bool((renumber.coherent_SAFE_REMOVE_delta!=0).any())

component_median_delta=int(lookup.loc['component_id median substitution','coherent_SAFE_REMOVE_delta'])
depth_delta=int(lookup.loc['mask_depth only','coherent_SAFE_REMOVE_delta'])
boundary_delta=int(lookup.loc['boundary_distance only','coherent_SAFE_REMOVE_delta'])
depth_boundary_delta=int(lookup.loc['depth + distance','coherent_SAFE_REMOVE_delta'])
three_delta=int(lookup.loc['original three-feature group','coherent_SAFE_REMOVE_delta'])

verdict={
 'page':PAGE,'read_only':True,'threshold_unchanged':REMOVE_THRESHOLD,
 'component_membership_hash_unchanged':membership_hash()==BASE_MEMBERSHIP_HASH,
 'any_raw_score_change_under_semantically_equivalent_renumbering':any_raw_change,
 'any_threshold_decision_change_under_semantically_equivalent_renumbering':any_threshold_change,
 'any_coherent_SAFE_REMOVE_change_under_semantically_equivalent_renumbering':any_coherent_change,
 'ordering_dependence_demonstrated':bool(any_raw_change or any_threshold_change or any_coherent_change),
 'component_id_median_delta':component_median_delta,
 'mask_depth_only_delta':depth_delta,
 'boundary_distance_only_delta':boundary_delta,
 'depth_plus_distance_delta':depth_boundary_delta,
 'original_three_feature_delta':three_delta,
 'original_three_feature_replays_164184':original_group_replay_exact,
 'interpretation':(
   'Frozen 59R1 is not invariant to arbitrary component numbering.' if (any_raw_change or any_threshold_change or any_coherent_change)
   else 'No ordering dependence was demonstrated on Page258 by the tested renumberings.'
 ),
 'causal_limit':'The audit proves sensitivity to identifier relabelling if decisions change. It does not claim the model uses no legitimate geometry.',
 'production_authority':False,
}
atomic_json(RUN/'invariance_verdict.json',verdict)
print(json.dumps(verdict,indent=2))

## 10. Compact non-overlay diagnostics

In [ ]:
plot=headline.copy()
plt.figure(figsize=(11,6))
plt.barh(plot.Audit,plot['Coherent SAFE_REMOVE delta'])
plt.axvline(0,color='black',linewidth=1)
plt.xlabel('Change in coherent SAFE_REMOVE pixels')
plt.title('Page258: disaggregated geometry and component-ID counterfactuals')
plt.tight_layout();plt.savefig(VISUALS/'page258_disaggregated_safe_remove_delta.png',dpi=180);plt.close()

plt.figure(figsize=(10,6))
plt.scatter(perm.mean_raw_score_movement_baseline_M5,perm.coherent_SAFE_REMOVE_delta)
for _,r in perm.iterrows(): plt.annotate(str(int(r.seed)),(r.mean_raw_score_movement_baseline_M5,r.coherent_SAFE_REMOVE_delta),fontsize=8)
plt.xlabel('Mean raw-score movement on baseline M5')
plt.ylabel('Coherent SAFE_REMOVE delta')
plt.title('Page258: fixed-seed component-ID permutations')
plt.tight_layout();plt.savefig(VISUALS/'page258_component_id_permutation_stability.png',dpi=180);plt.close()

## 11. Verify source immutability and package results

In [ ]:
source_hashes_after={k:sha256_file(v) for k,v in paths.items()}
require(source_hashes_after==source_hashes,'A source file changed during the read-only audit.')
atomic_json(AUDIT/'source_hashes_after.json',{k:{'path':str(paths[k]),'sha256':v} for k,v in source_hashes_after.items()})

summary_lines=[
 'PAGE258 COMPONENT-RENUMBERING INVARIANCE AUDIT', '',
 f'60R3 source: {SOURCE_60R3}', f'60R2 source: {SOURCE_60R2}', f'59R1 source: {REMOVE_RUN}',
 f'Frozen threshold: {REMOVE_THRESHOLD}', f'Components: {len(components)}',
 f'Ordering dependence demonstrated: {verdict["ordering_dependence_demonstrated"]}',
 f'Component-ID median SAFE_REMOVE delta: {component_median_delta}',
 f'Mask-depth-only SAFE_REMOVE delta: {depth_delta}',
 f'Boundary-distance-only SAFE_REMOVE delta: {boundary_delta}',
 f'Depth-plus-distance SAFE_REMOVE delta: {depth_boundary_delta}',
 f'Original three-feature SAFE_REMOVE delta: {three_delta}',
 f'Original group exactly replays 164184: {original_group_replay_exact}', '',
 verdict['interpretation'], verdict['causal_limit'],
 'No source artefact, threshold, model, calibrator, or Page258 baseline decision was modified.'
]
(RUN/'executive_summary.txt').write_text(chr(10).join(summary_lines)+chr(10))

curated=[
 RUN/'executive_summary.txt',RUN/'invariance_verdict.json',
 TABLES/'page258_geometry_disaggregation_table.csv',TABLES/'all_counterfactual_results.csv',
 TABLES/'component_id_permutation_replicates.csv',TABLES/'page258_component_membership.csv',
 AUDIT/'source_hashes_before.json',AUDIT/'source_hashes_after.json',
 VISUALS/'page258_disaggregated_safe_remove_delta.png',VISUALS/'page258_component_id_permutation_stability.png',
]
manifest=[]
for src in curated:
    require(src.is_file(),f'Missing curated output: {src}')
    rel=src.relative_to(RUN);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(src,dst)
    manifest.append({'relative_path':str(rel),'bytes':src.stat().st_size,'sha256':sha256_file(src)})
pd.DataFrame(manifest).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('60R3.1 Page258 read-only component-renumbering invariance audit.'+chr(10)+'The frozen model, calibrator, threshold and 60R2 outputs were not modified.'+chr(10))
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(sorted(str(p.relative_to(UPLOAD)) for p in UPLOAD.rglob('*') if p.is_file()))+chr(10))
zip_path=RUN/'60R3_1_Page258_Component_Renumbering_Invariance_Audit_upload_bundle.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(UPLOAD.rglob('*')):
        if p.is_file():zf.write(p,p.relative_to(UPLOAD))
require(zip_path.is_file() and zip_path.stat().st_size>0,'ZIP creation failed.')
print(chr(10).join(summary_lines));print('Upload ZIP:',zip_path)